In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# Resolve dataset robustly even if kernel starts from a different working directory.
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
candidate_paths = []
for root in search_roots:
    candidate_paths.extend([
        root / 'data' / 'cs-training.csv',
        root / 'credit-risk-api' / 'data' / 'cs-training.csv',
    ])

# Absolute fallback for this workspace on this machine.
candidate_paths.append(Path('C:/Users/Admin/credit-risk-api/data/cs-training.csv'))

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(f'Could not find cs-training.csv. Current working directory: {cwd}')

df = pd.read_csv(data_path)
print(f'Current working directory: {cwd}')
print(f'Loaded dataset from: {data_path.resolve()}')
print(f'Shape: {df.shape}')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\cs-test.csv'

In [ ]:
# 1) Inspect dataset
print('df.info():')
df.info()

print('\n\ndf.describe():')
display(df.describe(include='all'))

print('\n\ndf.isnull().sum():')
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# 2) Handle missing values: median imputation for numerical columns
num_cols = df.select_dtypes(include=['number']).columns

df[num_cols] = df[num_cols].apply(lambda col: col.fillna(col.median()))

print('Missing values after median imputation (numeric columns):')
display(df[num_cols].isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# 3) Plot class distribution (default vs non-default)
possible_targets = ['SeriousDlqin2yrs', 'default', 'is_default', 'target', 'loan_status']
target_col = next((c for c in possible_targets if c in df.columns), None)

if target_col is None:
    binary_cols = [c for c in df.columns if df[c].dropna().nunique() == 2]
    if not binary_cols:
        raise ValueError('No target column found. Please set target_col manually.')
    target_col = binary_cols[0]

print(f'Using target column: {target_col}')
print('Class counts:')
display(df[target_col].value_counts())
print('Class proportions:')
display(df[target_col].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df)
plt.title('Class Distribution: Default vs Non-Default')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# 4) Correlation heatmap (numeric features)
plt.figure(figsize=(12, 9))
corr = df.select_dtypes(include=['number']).corr()
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.2)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# 5) Engineer FICO bucket feature
fico_candidates = [c for c in df.columns if 'fico' in c.lower() or 'creditscore' in c.lower() or 'credit_score' in c.lower()]

if fico_candidates:
    fico_col = fico_candidates[0]
    fico_series = df[fico_col]
    print(f'Using existing FICO column: {fico_col}')
else:
    # This dataset has no native FICO score; build a proxy score from risk signals.
    risk_pct = (
        0.6 * df['RevolvingUtilizationOfUnsecuredLines'].rank(pct=True)
        + 0.4 * df['DebtRatio'].rank(pct=True)
    )
    df['fico_proxy'] = (850 - (risk_pct * 550)).clip(300, 850)
    fico_col = 'fico_proxy'
    fico_series = df[fico_col]
    print('No FICO column found. Created fico_proxy in range 300-850.')

fico_bins = [300, 580, 670, 740, 800, 851]
fico_labels = ['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional']

df['fico_bucket'] = pd.cut(fico_series, bins=fico_bins, labels=fico_labels, right=False)

display(df[[fico_col, 'fico_bucket']].head())
print('FICO bucket distribution:')
display(df['fico_bucket'].value_counts(dropna=False))

In [ ]:
# 6) Split into train/test (80/20, stratified)
feature_df = df.copy()

# Remove obvious non-feature columns if present.
for c in ['Unnamed: 0']:
    if c in feature_df.columns:
        feature_df = feature_df.drop(columns=[c])

X = feature_df.drop(columns=[target_col])
y = feature_df[target_col]

# One-hot encode categoricals (includes fico_bucket).
X = pd.get_dummies(X, drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shapes:', X_train.shape, y_train.shape)
print('Test shapes:', X_test.shape, y_test.shape)
print('Train class distribution before SMOTE:')
display(y_train.value_counts(normalize=True))

In [ ]:
# 7) Apply SMOTE on training set only
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print('Before SMOTE:')
display(y_train.value_counts())
print('After SMOTE:')
display(pd.Series(y_res).value_counts())
print('Resampled train shape:', X_res.shape, pd.Series(y_res).shape)

In [ ]:
# 8) Train baseline classifier and evaluate on original test set
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# liblinear is typically faster/stable for this binary baseline on tabular data.
baseline_model = LogisticRegression(max_iter=200, solver='liblinear', random_state=42)
baseline_model.fit(X_res, y_res)

y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print(f'Baseline Logistic Regression ROC-AUC: {roc_auc:.4f}')
print(f'Baseline Logistic Regression F1-score: {f1:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))